# Исследование надежности заемщиков


Во второй части проекта вы выполните шаги 3 и 4. Их вручную проверит ревьюер.
Чтобы вам не пришлось писать код заново для шагов 1 и 2, мы добавили авторские решения в ячейки с кодом. 



## Откройте таблицу и изучите общую информацию о данных

**Задание 1. Импортируйте библиотеку pandas. Считайте данные из csv-файла в датафрейм и сохраните в переменную `data`. Путь к файлу:**

`/datasets/data.csv`

In [1]:
import pandas as pd

try:
    data = pd.read_csv('/datasets/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

**Задание 2. Выведите первые 20 строчек датафрейма `data` на экран.**

In [2]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


**Задание 3. Выведите основную информацию о датафрейме с помощью метода `info()`.**

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


## Предобработка данных

### Удаление пропусков

**Задание 4. Выведите количество пропущенных значений для каждого столбца. Используйте комбинацию двух методов.**

In [4]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

**Задание 5. В двух столбцах есть пропущенные значения. Один из них — `days_employed`. Пропуски в этом столбце вы обработаете на следующем этапе. Другой столбец с пропущенными значениями — `total_income` — хранит данные о доходах. На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца `income_type`. Например, у человека с типом занятости `сотрудник` пропуск в столбце `total_income` должен быть заполнен медианным доходом среди всех записей с тем же типом.**

In [5]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

### Обработка аномальных значений

**Задание 6. В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке. таким артефактом будет отрицательное количество дней трудового стажа в столбце `days_employed`. Для реальных данных это нормально. Обработайте значения в этом столбце: замените все отрицательные значения положительными с помощью метода `abs()`.**

In [6]:
data['days_employed'] = data['days_employed'].abs()

**Задание 7. Для каждого типа занятости выведите медианное значение трудового стажа `days_employed` в днях.**

In [7]:
data.groupby('income_type')['days_employed'].agg('median')

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64

У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставьте их как есть. Тем более этот столбец не понадобится вам для исследования.

**Задание 8. Выведите перечень уникальных значений столбца `children`.**

In [8]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

**Задание 9. В столбце `children` есть два аномальных значения. Удалите строки, в которых встречаются такие аномальные значения из датафрейма `data`.**

In [9]:
data = data[(data['children'] != -1) & (data['children'] != 20)]

**Задание 10. Ещё раз выведите перечень уникальных значений столбца `children`, чтобы убедиться, что артефакты удалены.**

In [10]:
data['children'].unique()

array([1, 0, 3, 2, 4, 5])

### Удаление пропусков (продолжение)

**Задание 11. Заполните пропуски в столбце `days_employed` медианными значениями по каждого типа занятости `income_type`.**

In [11]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

**Задание 12. Убедитесь, что все пропуски заполнены. Проверьте себя и ещё раз выведите количество пропущенных значений для каждого столбца с помощью двух методов.**

In [12]:
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

### Изменение типов данных

**Задание 13. Замените вещественный тип данных в столбце `total_income` на целочисленный с помощью метода `astype()`.**

In [13]:
data['total_income'] = data['total_income'].astype(int)

### Обработка дубликатов

**Задание 14. Обработайте неявные дубликаты в столбце `education`. В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв. Приведите их к нижнему регистру. Проверьте остальные столбцы.**

In [14]:
data['education'] = data['education'].str.lower()

**Задание 15. Выведите на экран количество строк-дубликатов в данных. Если такие строки присутствуют, удалите их.**

In [15]:
data.duplicated().sum()

71

In [16]:
data = data.drop_duplicates()

### Категоризация данных

**Задание 16. На основании диапазонов, указанных ниже, создайте в датафрейме `data` столбец `total_income_category` с категориями:**

- 0–30000 — `'E'`;
- 30001–50000 — `'D'`;
- 50001–200000 — `'C'`;
- 200001–1000000 — `'B'`;
- 1000001 и выше — `'A'`.


**Например, кредитополучателю с доходом 25000 нужно назначить категорию `'E'`, а клиенту, получающему 235000, — `'B'`. Используйте собственную функцию с именем `categorize_income()` и метод `apply()`.**

In [17]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass

In [18]:
data['total_income_category'] = data['total_income'].apply(categorize_income)

**Задание 17. Выведите на экран перечень уникальных целей взятия кредита из столбца `purpose`.**

In [19]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

**Задание 18. Создайте функцию, которая на основании данных из столбца `purpose` сформирует новый столбец `purpose_category`, в который войдут следующие категории:**

- `'операции с автомобилем'`,
- `'операции с недвижимостью'`,
- `'проведение свадьбы'`,
- `'получение образования'`.

**Например, если в столбце `purpose` находится подстрока `'на покупку автомобиля'`, то в столбце `purpose_category` должна появиться строка `'операции с автомобилем'`.**

**Используйте собственную функцию с именем `categorize_purpose()` и метод `apply()`. Изучите данные в столбце `purpose` и определите, какие подстроки помогут вам правильно определить категорию.**

In [20]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'

In [21]:
data['purpose_category'] = data['purpose'].apply(categorize_purpose)

### Шаг 3. Исследуйте данные и ответьте на вопросы

#### 3.1 Есть ли зависимость между количеством детей и возвратом кредита в срок?

In [22]:
#функция для построение группированной таблицы
def make_grouped(df, column):
    new_df = df.groupby(column).agg({'debt':['count', 'sum']})
    new_df['debt_ratio'] = new_df['debt']['sum'] / new_df['debt']['count']
    return new_df

In [23]:
#группируем по столбцу 'children'
children_grouped = make_grouped(data, 'children')
children_grouped

debt       debt_ratio
          count   sum           
children                        
0         14091  1063   0.075438
1          4808   444   0.092346
2          2052   194   0.094542
3           330    27   0.081818
4            41     4   0.097561
5             9     0   0.000000

**Вывод:** 

В построенной таблице будем учитывать только категории с 0, 1 и 2 детьми, так как остальные категории не сопоставимы по размеру. Можем увидеть, что кол-во должников возрастает с увеличением числа детей. Причём разница между группами 0 и 1 сравнительно большая (0.017), а между 1 и 2 небольшая (0.003). Итого, зависимость есть, но значение имеет скорее сам факт наличия детей, нежели их количество.

#### 3.2 Есть ли зависимость между семейным положением и возвратом кредита в срок?

In [24]:
#выведем уникальные значения в столбце family_status
data['family_status'].unique()

array(['женат / замужем', 'гражданский брак', 'вдовец / вдова',
       'в разводе', 'Не женат / не замужем'], dtype=object)

In [25]:
#группируем таблицу по столбцу family_status
family_grouped = make_grouped(data, 'family_status')
family_grouped

debt      debt_ratio
                       count  sum           
family_status                               
Не женат / не замужем   2796  273   0.097639
в разводе               1189   84   0.070648
вдовец / вдова           951   63   0.066246
гражданский брак        4134  385   0.093130
женат / замужем        12261  927   0.075606

По этой таблице можно предварительно сказать, что больше всего должников со значением 'Не женат / не замужем', а меньше всего 'вдовец / вдова', однако значения status_family можно объединить в две группы: 'был в браке' и 'не был в браке'.

In [26]:
#создаём копию data
family_grouped = data.copy()

#меняем значения family_status, объединяя в две группы: 'есть парнёр' и 'одинок'
family_grouped.loc[(family_grouped['family_status'] == 'женат / замужем') |
                  (family_grouped['family_status'] == 'в разводе') |
                   (family_grouped['family_status'] == 'вдовец / вдова'),
                   'family_status'] = 'был в браке'
family_grouped.loc[(family_grouped['family_status'] == 'Не женат / не замужем') |
                  (family_grouped['family_status'] == 'гражданский брак'),
                   'family_status'] = 'не был в браке'

#строим новую группированную таблицу
family_grouped = make_grouped(family_grouped, 'family_status')
family_grouped

debt       debt_ratio
                count   sum           
family_status                         
был в браке     14401  1074   0.074578
не был в браке   6930   658   0.094949

**Вывод:** 

Поделив значения family_status на 'был в браке' и 'не был в браке', видим, что доля должников во второй категории больше, следовательно у заёмщиков бывших когда-либо в официальном браке больше вероятность вернуть кредит в срок.

#### 3.3 Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

In [27]:
#сгрупперуем по столбцу total_income_category и найдём count (общее кол-во) и sum (кол-во дожников)
income_grouped = data.groupby('total_income_category').agg({'debt' : ['count', 'sum']})

#поделим кол-во дожников на общее число и добавим в качестве отдельного столбца
income_grouped['debt_ratio'] = income_grouped['debt']['sum'] / income_grouped['debt']['count']
income_grouped

debt       debt_ratio
                       count   sum           
total_income_category                        
A                         25     2   0.080000
B                       5014   354   0.070602
C                      15921  1353   0.084982
D                        349    21   0.060172
E                         22     2   0.090909

**Вывод:** 

В построенной таблице будем рассматривать только группы B и C, так как остальные группы не сопоставимы по размеру. Так, можем сказать, что у заёмщиков с большим доходом больше вероятность вернуть кредит в срок.

#### 3.4 Как разные цели кредита влияют на его возврат в срок?

In [28]:
#сгрупперуем по столбцу total_income_category и найдём count (общее кол-во) и sum (кол-во дожников)
purpose_grouped = data.groupby('purpose_category').agg({'debt' : ['count', 'sum']})

#поделим кол-во дожников на общее число и добавим в качестве отдельного столбца
purpose_grouped['debt_ratio'] = purpose_grouped['debt']['sum'] / purpose_grouped['debt']['count']
purpose_grouped

debt      debt_ratio
                          count  sum           
purpose_category                               
операции с автомобилем     4279  400   0.093480
операции с недвижимостью  10751  780   0.072551
получение образования      3988  369   0.092528
проведение свадьбы         2313  183   0.079118

**Вывод:** 

По построенной таблице видно, что в строках со значениями 'операции с недвижимостью' и 'проведение свадьбы' доля должников меньше, а в строках со значениями 'операции с автомобилем' и 'получение образования' больше. Скорее всего это связано с характерами самих целей.
- Автомобиль после приобретения нужно обслуживать и он может сломаться.
- Образование - это инвестиция в будущее, которая может и не окупиться.
- Недвижимость достаточно стабильное вложение.
- Свадьба - социально важное событие, что может обеспечивать давление на заёмщика, способствующее возврату в срок. С другой стороны много на чём можно сэкономить, что может снизить сумму кредита и также повысить шанс возврата в срок.

#### 3.5 Приведите возможные причины появления пропусков в исходных данных.

*Ответ:* 

Данные о стаже работы и уровне дохода являются обязательными для заполнения при взятии кредита, следовательно причинами появления пропусков в исходных данных скорее всего являются человеческий фактор или техническая ошибка.

#### 3.6 Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

*Ответ:* 

Заполнить пропуски медианным значением — лучшее решение для количественных переменных так как при заполнении средним арифметиеским сильно выделяющиеся значения могут значительно искажать результат, чего не происходит при использовании медианы.

### Шаг 4: общий вывод.

По полученным данным о кредиторах была проведена следующая работа:

Предобработка данных:
1. Пропуски в столбцах days_employed и total_income были заменены на медианные значения соответствующих income_type.
2. Аномальные значения в столбце days_employed (отрицательные) были заменены на их модуль.
3. Строки с аномальными значениями в столбце children (-1 и 20) были удалены.
4. Тип данных в столбце total_income был изменён со строкового на целочисленный.
5. Удалены неявные дубликаты в столбце education
6. Произведена категоризация в столбцах total_income и purpose.

Выявленные зависимости просрочки от разных факторов:
1. Заёмщики, у которых нет детей, с большей вероятностью вернут кредит в срок. При увеличения числа детей у заёмщика шанс возврата в срок уменьшается, но незначительно.
2. Заёмщики, состоявшие когда-либо в официальном браке, с большей вероятностью вернут кредит в срок, чем те, кто в нём никогда не состоял.
3. Чем больше у заёмщика общий доход, тем больше вероятность возврата кредита в срок.
4. Кредиты взятые для проведения операций с недвижимостью и проведения свадьбы с большей вероятностью будут возвращены в срок, нежели кредиты взятые для получения образования или проведения операций с автомобилем.

Портрет идельного заёмщика:
- Нет детей
- Состоял или состоит в официальном браке
- Имеет большой доход
- Цель кредита связана с недвижимостью или проведением свадьбы

Портрет рискованного заёмщика:
- Много детей
- Никогда не состоял в официальном браке
- Имеет маленький доход
- Цель кредита связана с получением образования или проведением операций с автомобилем

Рекомендации заказчику:

Создать различные специальные условия для заёмщиков попадающие под образы надёжного и рискованного заёмщика соответственно
- Для надёжных заёмщиков предлагать пониженную кредитную ставку, льготы на ипотеку и свадебные кредиты, тем самым больше их привлекая.
- Для рискованных заказчиков уменьшать сумму кредита, повышать кредитную ставку, требовать дополнительных гарантий (залог).

Также можно провести дополнительные исследования, изучить другие факторы заёмщиков (регион, возраст и т.п.), для уточнения образов надёжного и рискованного заёмщиков.